### Loading Libraries

In [1]:
import os
from dotenv import load_dotenv
import praw
from openai import OpenAI
import pandas as pd
import time
from pinecone import Pinecone

### Connect APIs

In [52]:
### Setup
load_dotenv('../.env.local')
REDDIT_CLIENT_ID = os.environ.get('REDDIT_CLIENT_ID')
REDDIT_SECRET_ID = os.environ.get('REDDIT_SECRET_ID')
OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY')
PINECONE_API_KEY = os.environ.get('PINECONE_API_KEY')


In [53]:
### Reddit API
reddit = praw.Reddit(
    client_id = REDDIT_CLIENT_ID,
    client_secret = REDDIT_SECRET_ID,
    user_agent = "researcher"
)
# Session options: controversial, gilded, hot, new, rising, top
print(reddit.read_only)
for submission in reddit.subreddit("travel").hot(limit=3):
    print(submission.title)

### OpenAI API
client = OpenAI(api_key=OPENAI_API_KEY)

### Pinecone API
pc = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index('flight-reviews')

True
Reminder: any use of ChatGPT or AI tools will result in a ban
Nepal - On the way to the Annapurna Base Camp
Székesfehérvár, former capital of Hungary


### Scrape Reddit

In [ ]:
### get comments and posts
target_subreddits = reddit.subreddit("delta+united+AmericanAirlines+travel+awardtravel")
search_query = ''' 
                ("Business Class" OR "First Class" OR "Status" OR "Priority") AND 
                ("App" OR "Website" OR "Boarding" OR "Wifi" OR "Check-in" OR "Glitch") 
                '''

title, comment, score = [], [], []
for submission in target_subreddits.search(
    search_query,
    sort="relevance",
    time_filter="year",
    limit=1):
    print(submission.title)
    print("comments:")
    submission.comments.replace_more(limit=1)
    for i in range(len(submission.comments.list())):
        print(submission.comments.list()[i].body)
        print(submission.score)
        print("--")
        title.append(submission.title)
        comment.append(submission.comments.list()[i].body)
        score.append(submission.score)

        # time.sleep()

In [33]:
reddit_df = pd.DataFrame({
    'Title': title,
    'Comment': comment,
    'Score': score
})
display(reddit_df)

,Title,Comment,Score
0,Passenger tried to pull rank on me,LOL I always fly bulkhead. If wishes were fish...,12525
1,Passenger tried to pull rank on me,"This reads like chatGPT, 100%",12525
2,Passenger tried to pull rank on me,[deleted],12525
3,Passenger tried to pull rank on me,And then everyone clapped and carried me out o...,12525
4,Passenger tried to pull rank on me,While I always appreciate a good bit of fictio...,12525
...,...,...,...
8973,My First Class Seat Squatter,Let’s argue more about a beer on an airline. Y...,1711
8974,My First Class Seat Squatter,Can you point to where someone said your exper...,1711
8975,My First Class Seat Squatter,Please tell more about what I meant to say… I ...,1711
8976,My First Class Seat Squatter,I'm not sure why you're so aggressive here? Ho...,1711


### Vectorize Comments

In [ ]:
# Filter out empty/removed comments
clean_comments = [c for c in comment if c not in ['[removed]', '[deleted]'] and len(c.strip()) > 0]
print(f"Total comments to embed: {len(clean_comments):,}")

Total comments to embed: 557


##### Batch Embeddings

In [60]:
# Set batch and comment char length
batch_size = 200
max_chars = 200000

all_vectors = []
total_batches = (len(clean_comments) + batch_size - 1) // batch_size
print(f'Processing {total_batches} batches of up to {batch_size} comments')

for batch_num in range(0, len(clean_comments), batch_size):
    batch = clean_comments[batch_num:batch_num + batch_size]
    batch_char = sum(len(c) for c in batch)

    print(f'Batch {(batch_num // batch_size) + 1}/{total_batches}: Embedding {len(batch)} comments ({batch_char:,} chars...', end=" ")

    embeddings_response = client.embeddings.create(
        model="text-embedding-3-small",
        input=batch
    )

    vectors = [
        (f'id_{batch_num}_{j}', embeddings_response.data[j].embedding, {'text': batch[j]})
        for j in range(len(batch))
    ]

    all_vectors.extend(vectors)
    print(f'Done ({len(all_vectors):,} vectors total)')

Processing 3 batches of up to 200 comments
Batch 1/3: Embedding 200 comments (21,159 chars... Done (200 vectors total)
Batch 2/3: Embedding 200 comments (21,782 chars... Done (400 vectors total)
Batch 3/3: Embedding 157 comments (19,540 chars... Done (557 vectors total)


##### Upsert to Pinecone

In [65]:
upsert_batch_size = 100
for i in range(0, len(all_vectors), upsert_batch_size):
    batch = all_vectors[i:i+upsert_batch_size]
    index.upsert(vectors=batch)
    print(f'Uploading {i + len(batch)}/{len(all_vectors)} vectors')
print(f'Successfully uploaded {len(all_vectors)} vectors to Pinecone')

Uploading 100/557 vectors
Uploading 200/557 vectors
Uploading 300/557 vectors
Uploading 400/557 vectors
Uploading 500/557 vectors
Uploading 557/557 vectors
Successfully uploaded 557 vectors to Pinecone


### Query Relevant Comments

In [ ]:
query = "Business class flyers care about their online boarding experience."
query_embedding = client.embeddings.create(
    model="text-embedding-3-small",
    input=query
    ).data[0].embedding

results = index.query(vector=query_embedding, top_k=3, include_metadata=True)

for match in results['matches']:
    print(f'Score: {match['score']:.3f}')
    print(f'Comment {match['metadata']['text']}\n')

Score: 0.477
Comment BS ... she complains about a window seat when you are in aisle?  C is aisle.  A or F (midsize planes) are window.  BUT I have experienced people mixing up the row numbers... until they pull out their boarding pass 🤪

Score: 0.455
Comment Do such people even think twice before talking or doing something before embarrassing themselves? 

Most of us are frequent flier in some form and these privilege status given by airlines don’t do much these days and yet people pride themselves about being a certain status. 

Gives me chills everytime I have to take an international flight

Score: 0.447
Comment I always fly private. All of you other passengers will need to deplane now.

{'matches': [{'id': 'id_400_80',
              'metadata': {'text': 'BS ... she complains about a window seat '
                                   'when you are in aisle?  C is aisle.  A or '
                                   'F (midsize planes) are window.  BUT I have '
                           